# MIMIC-CXR MedSigLIP embeddings

This notebook demonstrates how to use the clean MIMIC-CXR dataset with MedSigLIP. It reads the split CSV, selects example chest X-ray images from the dataset, extracts image and text embeddings, compares them with simple zero-shot prompts, and visualizes the embeddings with PCA.

Use it as a compact reference for turning the clean dataset into embeddings that can later be used for retrieval, zero-shot experiments, or training a small downstream model.

----

Este notebook muestra cómo usar el dataset MIMIC-CXR con MedSigLIP. Lee el CSV del split, selecciona ejemplos de radiografías de tórax desde el dataset, extrae embeddings de imagen y texto, los compara con prompts zero-shot simples y visualiza los embeddings con PCA.

Úsalo como una referencia compacta para convertir el dataset limpio en embeddings que luego pueden usarse para recuperación, experimentos zero-shot o entrenamiento de un modelo downstream pequeño.


In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
import torch
from transformers import AutoModel, AutoProcessor

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_ID = "google/medsiglip-448"
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(MODEL_ID, token=HF_TOKEN).to(device)
model.eval()


device: cpu


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 1152)
      (position_embedding): Embedding(64, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304,

In [3]:
# Reemplazar con tu ruta al dataset
#DATASET_DIR = Path("PATH-TO-DATASET/MIMIC-CXR")
DATASET_DIR = Path(os.environ["SCRATCH"]) / "datasets" / "datatondatasets" / "MIMIC-CXR"

labels = pd.read_csv(DATASET_DIR / "train.csv")
print(labels.shape)
labels.head()


(101138, 37)


,image,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,split,ViewPosition,Rows,Columns,StudyDate,...,Support Devices,race,sex,age,anchor_year,anchor_year_group,dod,race_label,sex_label,report
0,s51321189_d85c9f15-f0f84927-761f30e0-51c2d319-...,d85c9f15-f0f84927-761f30e0-51c2d319-f2d917f0,19702416,51321189,CHEST (PORTABLE AP),train,AP,3056,2544,21680821,...,NaN,White,Male,91.0,2168,2011 - 2013,2168-11-01,0,0,FINAL REPORT\...
1,s51292704_0024603b-12db30e2-ab32c9cb-dae5a3fc-...,0024603b-12db30e2-ab32c9cb-dae5a3fc-b2c598b4,13339704,51292704,CHEST (PORTABLE AP),train,AP,2544,3056,21360418,...,1.0,Black,Male,68.0,2133,2011 - 2013,NaN,2,0,FINAL REPORT\...
2,s54048859_8a4aaaee-55fcf98f-a036a8e7-da71eed1-...,8a4aaaee-55fcf98f-a036a8e7-da71eed1-4d7bf032,12668169,54048859,CHEST (PORTABLE AP),train,AP,2820,2539,21560819,...,1.0,White,Male,54.0,2156,2011 - 2013,2156-08-19,0,0,FINAL REPORT\...
3,s58144222_9886b0fe-9121c65e-c8d74649-4b88c530-...,9886b0fe-9121c65e-c8d74649-4b88c530-9b3943fb,10309415,58144222,CHEST (PORTABLE AP),train,AP,2544,3056,21140603,...,NaN,White,Male,88.0,2114,2014 - 2016,NaN,0,0,FINAL REPORT\...
4,s59315061_61b65859-4e25d250-d3faadc7-0fda22dd-...,61b65859-4e25d250-d3faadc7-0fda22dd-14abe901,19504029,59315061,CHEST (PA AND LAT),train,PA,3056,2544,21780421,...,NaN,Black,Female,54.0,2178,2011 - 2013,NaN,2,1,FINAL REPORT\...


In [4]:
def load_rgb(path):
    return Image.open(path).convert("RGB")


def run_medsiglip(images, texts):
    inputs = processor(
        text=texts,
        images=images,
        padding="max_length",
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits_per_image, dim=1).detach().cpu().numpy()
    image_embeds = outputs.image_embeds.detach().cpu().numpy()
    text_embeds = outputs.text_embeds.detach().cpu().numpy()
    return probs, image_embeds, text_embeds


def show_predictions(images, texts, probs):
    for image_index, image in enumerate(images):
        display(image)
        for text, score in zip(texts, probs[image_index]):
            print(f"{score:.2%} - {text}")
        print()


def plot_embedding_pca(image_embeds, text_embeds, image_labels, text_labels, title):
    vectors = np.vstack([image_embeds, text_embeds])
    coords = PCA(n_components=2, random_state=0).fit_transform(vectors)
    n_images = len(image_embeds)

    plt.figure(figsize=(7, 5))
    plt.scatter(coords[:n_images, 0], coords[:n_images, 1], label="images", s=90)
    plt.scatter(coords[n_images:, 0], coords[n_images:, 1], label="texts", marker="x", s=90)

    for label, (x, y) in zip(image_labels, coords[:n_images]):
        plt.text(x, y, label, fontsize=9)
    for label, (x, y) in zip(text_labels, coords[n_images:]):
        plt.text(x, y, label, fontsize=9)

    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [5]:
label_cols = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Lung Opacity", "No Finding", "Pleural Effusion", "Pneumonia",
    "Pneumothorax", "Support Devices",
]

sample = labels.sample(min(2, len(labels)), random_state=3).reset_index(drop=True)
images = [load_rgb(DATASET_DIR / "images" / row.image) for row in sample.itertuples()]
image_labels = [f"image {i}" for i in range(len(images))]

texts = [
    "a chest x-ray with no finding",
    "a chest x-ray with pneumonia",
    "a chest x-ray with pleural effusion",
    "a chest x-ray with cardiomegaly",
]

sample[["image", *[c for c in label_cols if c in sample.columns]]].head()


,image,Atelectasis,Cardiomegaly,Consolidation,Edema,Lung Opacity,No Finding,Pleural Effusion,Pneumonia,Pneumothorax,Support Devices
0,s50193671_a483c3dd-425e7fd8-65af5338-149579ff-...,NaN,0.0,NaN,NaN,1.0,NaN,0.0,NaN,0.0,NaN
1,s57895303_b3412098-4d7a3a55-2a980418-0354fe36-...,0.0,NaN,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN


In [ ]:
probs, image_embeds, text_embeds = run_medsiglip(images, texts)
show_predictions(images, texts, probs)
print("image embeddings:", image_embeds.shape)
print("text embeddings:", text_embeds.shape)


In [ ]:
plot_embedding_pca(
    image_embeds,
    text_embeds,
    image_labels=image_labels,
    text_labels=texts,
    title="MIMIC-CXR MedSigLIP embedding PCA",
)
